# 06 - Task 2 Experiments (T2-C1 to T2-C4)

Notebook ini menjalankan eksperimen Task 2 dengan empat konfigurasi:

| Config | Encoder | Fusion | Data | Tujuan |
|---|---|---|---|---|
| T2-C1 | fine-tuned | full `[x,y,x*y,|x-y|]` | baseline | baseline pembanding |
| T2-C2 | frozen | full | baseline | uji kontribusi encoder |
| T2-C3 | fine-tuned | basic `[x,y]` | baseline | uji fusion sederhana |
| T2-C4 | fine-tuned | full | pitch-shift holdout | uji generalisasi interval pitch |

Cara pakai:

1. Jalankan cell dari atas sampai cell evaluasi.
2. Ubah `CONFIG_ID` di cell konfigurasi untuk memilih C1, C2, C3, atau C4.
3. `CONFIG_ID="C1"` dan `REUSE_C1=True` akan memakai checkpoint `T2-baseline_seed42` tanpa retrain.
4. Untuk smoke test cepat, set `FAST_DEV=True`; hasil fast-dev bukan hasil final.
5. C2-C4 akan dilatih dari backbone `T1-C1_seed42` dan disimpan sebagai `T2-<CONFIG>_seed42`.

Semua konfigurasi dievaluasi pada test set penuh dan pada pitch-holdout test set yang sama.


In [ ]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data" / "processed").exists():
    raise RuntimeError(
        "Jalankan notebook dari root repo atau dari folder notebooks/. "
        f"Path saat ini: {ROOT}"
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.baseline_dorfer import (
    CLASS_NAMES,
    AudioEncoder,
    SheetEncoder,
    Task2Dataset,
    Task2Model,
    binary_metrics,
    classification_metrics,
    configure_torch,
    count_parameters,
    load_checkpoint,
    resolve_device,
    save_json,
    set_seed,
    to_jsonable,
    topk_accuracy,
)

DATA_DIR = ROOT / "data" / "processed"
print("Root:", ROOT)
print("Data:", DATA_DIR)


In [ ]:
CONFIG_ID = "C1"  # C1 | C2 | C3 | C4
SEED = 42
FAST_DEV = False
OVERWRITE = False
REUSE_C1 = True

if CONFIG_ID not in {"C1", "C2", "C3", "C4"}:
    raise ValueError(f"CONFIG_ID tidak dikenal: {CONFIG_ID}")

RUN_ID = f"T2-{CONFIG_ID}_seed{SEED}"
REPORTS_DIR = ROOT / "reports" / "experiments" / RUN_ID
FIG_DIR = REPORTS_DIR / "figures"
CKPT_DIR = ROOT / "checkpoints" / RUN_ID
for directory in (REPORTS_DIR, FIG_DIR, CKPT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

HISTORY_PATH = REPORTS_DIR / "history.csv"
METRICS_PATH = REPORTS_DIR / "metrics.json"
CONFIG_PATH = REPORTS_DIR / "config.json"
SOURCE_PATH = REPORTS_DIR / "source.json"

BASELINE_CKPT = ROOT / "checkpoints" / "T2-baseline_seed42" / "best.pt"
TASK1_CKPT = ROOT / "checkpoints" / "T1-C1_seed42" / "best.pt"

if not TASK1_CKPT.exists():
    raise FileNotFoundError(
        "Checkpoint backbone Task 1 belum ada: "
        f"{TASK1_CKPT}. Jalankan eksperimen T1-C1 terlebih dahulu."
    )

if CONFIG_ID == "C1" and REUSE_C1:
    if not BASELINE_CKPT.exists():
        raise FileNotFoundError(
            "Checkpoint T2-baseline_seed42 tidak ditemukan: "
            f"{BASELINE_CKPT}. Set REUSE_C1=False untuk retrain C1."
        )
    CHECKPOINT_PATH = BASELINE_CKPT
    SOURCE_RUN = "T2-baseline_seed42"
    SOURCE_CKPT = BASELINE_CKPT
else:
    CHECKPOINT_PATH = CKPT_DIR / "best.pt"
    SOURCE_RUN = None
    SOURCE_CKPT = None

if METRICS_PATH.exists() and not OVERWRITE and not (CONFIG_ID == "C1" and REUSE_C1):
    raise RuntimeError(
        f"Run {RUN_ID} sudah ada. Set OVERWRITE=True untuk menimpa, "
        "atau ganti CONFIG_ID/SEED."
    )

print("Config  :", CONFIG_ID)
print("Run ID  :", RUN_ID)
print("Reports :", REPORTS_DIR)
print("Ckpt    :", CHECKPOINT_PATH)
print("Source  :", SOURCE_RUN)


In [ ]:
if BASELINE_CKPT.exists():
    baseline_checkpoint = load_checkpoint(BASELINE_CKPT, map_location="cpu")
    BASE_CONFIG = dict(baseline_checkpoint["config"])
else:
    BASE_CONFIG = {
        "profile": "dorfer-full",
        "seed": SEED,
        "device": "auto",
        "threads": None,
        "embed_dim": 32,
        "channels": (24, 48, 96, 96),
        "hidden_dim": 128,
        "dropout": 0.2,
        "num_classes": 4,
        "batch_size": 256,
        "eval_batch_size": 512,
        "epochs": 8,
        "early_stop_patience": 3,
        "lr_encoder": 3e-4,
        "lr_head": 1e-3,
        "weight_decay": 1e-5,
        "grad_clip": 5.0,
        "freeze_sheet": False,
        "train_one_variant_per_source": True,
        "num_workers": 0,
        "fast_dev": False,
    }

BASE_CONFIG.setdefault("embed_dim", 32)
BASE_CONFIG.setdefault("channels", (24, 48, 96, 96))
BASE_CONFIG.setdefault("hidden_dim", 128)
BASE_CONFIG.setdefault("dropout", 0.2)
BASE_CONFIG.setdefault("num_classes", 4)
BASE_CONFIG.setdefault("batch_size", 256)
BASE_CONFIG.setdefault("eval_batch_size", 512)
BASE_CONFIG.setdefault("epochs", 8)
BASE_CONFIG.setdefault("early_stop_patience", 3)
BASE_CONFIG.setdefault("lr_encoder", 3e-4)
BASE_CONFIG.setdefault("lr_head", 1e-3)
BASE_CONFIG.setdefault("weight_decay", 1e-5)
BASE_CONFIG.setdefault("grad_clip", 5.0)
BASE_CONFIG.setdefault("train_one_variant_per_source", True)
BASE_CONFIG.setdefault("num_workers", 0)

VARIANT_DELTAS = {
    "C1": {"fusion": "full", "freeze_encoders": False, "data_mode": "baseline"},
    "C2": {"fusion": "full", "freeze_encoders": True, "data_mode": "baseline"},
    "C3": {"fusion": "basic", "freeze_encoders": False, "data_mode": "baseline"},
    "C4": {"fusion": "full", "freeze_encoders": False, "data_mode": "pitch_holdout"},
}

CONFIG = dict(BASE_CONFIG)
CONFIG.update(VARIANT_DELTAS[CONFIG_ID])
CONFIG["config_id"] = CONFIG_ID
CONFIG["run_id"] = RUN_ID
CONFIG["seed"] = SEED
CONFIG["fast_dev"] = bool(FAST_DEV)
CONFIG["reuse_c1"] = bool(CONFIG_ID == "C1" and REUSE_C1)
CONFIG["source_run"] = SOURCE_RUN
CONFIG["source_checkpoint"] = str(SOURCE_CKPT) if SOURCE_CKPT else None
CONFIG["task1_checkpoint"] = str(TASK1_CKPT)
CONFIG["pitch_split"] = {"train": [1, 6], "val": [7, 8], "test": [9, 12]}

set_seed(int(CONFIG["seed"]))
configure_torch(CONFIG.get("threads"))
DEVICE = resolve_device(CONFIG.get("device", "auto"))

print("Device:", DEVICE)
print(json.dumps(to_jsonable(CONFIG), indent=2))


In [ ]:
SPLITS = ["train", "val", "test"]
required = []
for split in SPLITS:
    required.extend([
        DATA_DIR / f"error_windows_{split}.npy",
        DATA_DIR / f"task2_windows_{split}.parquet",
        DATA_DIR / f"score_crops_{split}.npy",
        DATA_DIR / f"noteheads_{split}.parquet",
    ])

missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Artefak Task 2 belum lengkap:\n" + "\n".join(missing)
    )

meta_frames = {
    split: pd.read_parquet(DATA_DIR / f"task2_windows_{split}.parquet").reset_index(drop=True)
    for split in SPLITS
}

rows = []
for split in SPLITS:
    meta = meta_frames[split]
    rows.append({
        "split": split,
        "rows": len(meta),
        "source_windows": meta["source_window_id"].nunique(),
        "classes": meta["class_name"].value_counts().to_dict(),
        "soundfonts": meta["soundfont"].nunique(),
        "pieces": meta["piece"].nunique(),
        "work_groups": meta["work_group"].nunique(),
    })
display(pd.DataFrame(rows))

for split in SPLITS:
    variants = meta_frames[split].groupby("source_window_id").size()
    if not variants.eq(4).all():
        raise ValueError(f"{split}: ada source window tanpa 4 varian")
print("Setiap source window pada data mentah memiliki 4 varian.")


In [ ]:
def filter_pitch_meta(meta, split, mode, pitch_split):
    filtered = meta.copy()
    if mode != "pitch_holdout":
        return filtered.reset_index(drop=True)
    delta = filtered["pitch_delta"].abs()
    low, high = pitch_split[split]
    keep = (filtered["class_id"] != 1) | ((delta >= low) & (delta <= high))
    return filtered.loc[keep].reset_index(drop=True)

train_meta = filter_pitch_meta(
    meta_frames["train"], "train", CONFIG["data_mode"], CONFIG["pitch_split"]
)
val_meta = filter_pitch_meta(
    meta_frames["val"], "val", CONFIG["data_mode"], CONFIG["pitch_split"]
)
test_meta_full = meta_frames["test"].copy().reset_index(drop=True)
test_meta_pitch = filter_pitch_meta(
    meta_frames["test"], "test", "pitch_holdout", CONFIG["pitch_split"]
)

if CONFIG["data_mode"] == "pitch_holdout":
    train_wrong = train_meta.loc[train_meta["class_id"] == 1, "pitch_delta"].abs()
    val_wrong = val_meta.loc[val_meta["class_id"] == 1, "pitch_delta"].abs()
    test_wrong = test_meta_pitch.loc[test_meta_pitch["class_id"] == 1, "pitch_delta"].abs()
    assert (train_wrong <= 6).all(), "T2-C4 train masih punya |pitch_delta| > 6"
    assert val_wrong.between(7, 8).all(), "T2-C4 val bukan interval 7-8"
    assert test_wrong.between(9, 12).all(), "T2-C4 pitch test bukan interval 9-12"

def subsample_meta(meta, per_class, seed):
    parts = [
        group.sample(n=min(per_class, len(group)), random_state=seed)
        for _, group in meta.groupby("class_id")
    ]
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

if FAST_DEV:
    train_meta = subsample_meta(train_meta, 1000, SEED)
    val_meta = subsample_meta(val_meta, 250, SEED)
    test_meta_full = subsample_meta(test_meta_full, 500, SEED)
    test_meta_pitch = subsample_meta(test_meta_pitch, 500, SEED)
    print("FAST_DEV aktif: subset per kelas, 2 epoch. Hasil bukan hasil final.")

rows = []
for name, meta in [
    ("train", train_meta),
    ("val", val_meta),
    ("test_full", test_meta_full),
    ("test_pitch", test_meta_pitch),
]:
    rows.append({
        "split": name,
        "rows": len(meta),
        "source_windows": meta["source_window_id"].nunique(),
        "classes": meta["class_name"].value_counts().to_dict(),
    })
display(pd.DataFrame(rows))


In [ ]:
def make_task2_dataset(split, meta, one_variant_per_source):
    noteheads = pd.read_parquet(DATA_DIR / f"noteheads_{split}.parquet")
    return Task2Dataset(
        DATA_DIR / f"score_crops_{split}.npy",
        DATA_DIR / f"error_windows_{split}.npy",
        meta,
        noteheads,
        score_bank_indices=None,
        one_variant_per_source=one_variant_per_source,
    )

train_dataset = make_task2_dataset(
    "train", train_meta, CONFIG["train_one_variant_per_source"]
)
val_dataset = make_task2_dataset("val", val_meta, False)
test_full_dataset = make_task2_dataset("test", test_meta_full, False)
test_pitch_dataset = make_task2_dataset("test", test_meta_pitch, False)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    drop_last=False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["eval_batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
)
test_full_loader = DataLoader(
    test_full_dataset,
    batch_size=CONFIG["eval_batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
)
test_pitch_loader = DataLoader(
    test_pitch_dataset,
    batch_size=CONFIG["eval_batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
)

score_images, specs, labels = next(iter(train_loader))
print("score images:", tuple(score_images.shape), score_images.dtype)
print("audio       :", tuple(specs.shape), specs.dtype)
print("labels      :", tuple(labels.shape), labels.dtype, "| values:", labels.unique().tolist())
print(f"train items: {len(train_dataset):,} | val: {len(val_dataset):,}")
print(f"test full : {len(test_full_dataset):,} | test pitch: {len(test_pitch_dataset):,}")


In [ ]:
class BasicFusionClassifier(nn.Module):
    def __init__(self, embed_dim=32, hidden_dim=128, num_classes=4, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * embed_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, sheet_emb, audio_emb):
        x = F.normalize(sheet_emb, dim=1)
        y = F.normalize(audio_emb, dim=1)
        return self.net(torch.cat([x, y], dim=1))


class BasicFusionTask2Model(nn.Module):
    def __init__(
        self,
        sheet_encoder,
        audio_encoder,
        embed_dim=32,
        hidden_dim=128,
        num_classes=4,
        dropout=0.2,
    ):
        super().__init__()
        self.sheet = sheet_encoder
        self.audio = audio_encoder
        self.classifier = BasicFusionClassifier(
            embed_dim=embed_dim,
            hidden_dim=hidden_dim,
            num_classes=num_classes,
            dropout=dropout,
        )

    def forward(self, score_image, audio_spec):
        return self.classifier(self.sheet(score_image), self.audio(audio_spec))

    def forward_from_embeddings(self, score_emb, audio_emb):
        return self.classifier(score_emb, audio_emb)


def build_task2_model(fusion, freeze_encoders):
    task1_checkpoint = load_checkpoint(TASK1_CKPT, map_location="cpu")
    for key in ("sheet_encoder", "audio_encoder"):
        if key not in task1_checkpoint:
            raise KeyError(f"Checkpoint Task 1 tidak punya '{key}'")

    task1_channels = tuple(
        task1_checkpoint.get("config", {}).get("channels", CONFIG["channels"])
    )
    if task1_channels != tuple(CONFIG["channels"]):
        raise ValueError(
            "Channels Task 2 tidak cocok dengan checkpoint Task 1: "
            f"{tuple(CONFIG['channels'])} vs {task1_channels}"
        )

    sheet_encoder = SheetEncoder(
        embed_dim=CONFIG["embed_dim"], channels=CONFIG["channels"]
    )
    audio_encoder = AudioEncoder(
        embed_dim=CONFIG["embed_dim"], channels=CONFIG["channels"]
    )
    sheet_encoder.load_state_dict(task1_checkpoint["sheet_encoder"])
    audio_encoder.load_state_dict(task1_checkpoint["audio_encoder"])

    if fusion == "full":
        model = Task2Model(
            sheet_encoder=sheet_encoder,
            audio_encoder=audio_encoder,
            embed_dim=CONFIG["embed_dim"],
            hidden_dim=CONFIG["hidden_dim"],
            num_classes=CONFIG["num_classes"],
            dropout=CONFIG["dropout"],
        )
    elif fusion == "basic":
        model = BasicFusionTask2Model(
            sheet_encoder=sheet_encoder,
            audio_encoder=audio_encoder,
            embed_dim=CONFIG["embed_dim"],
            hidden_dim=CONFIG["hidden_dim"],
            num_classes=CONFIG["num_classes"],
            dropout=CONFIG["dropout"],
        )
    else:
        raise ValueError(f"Fusion tidak dikenal: {fusion}")

    if freeze_encoders:
        for parameter in model.sheet.parameters():
            parameter.requires_grad = False
        for parameter in model.audio.parameters():
            parameter.requires_grad = False

    return model.to(DEVICE)

model = build_task2_model(CONFIG["fusion"], CONFIG["freeze_encoders"])
print(f"Total parameters    : {count_parameters(model):,}")
print(f"Trainable parameters: {count_parameters(model, trainable_only=True):,}")
print("Fusion              :", CONFIG["fusion"])
print("Freeze encoders     :", CONFIG["freeze_encoders"])
print(model)


In [ ]:
criterion = nn.CrossEntropyLoss()
parameter_groups = [
    {"params": model.classifier.parameters(), "lr": CONFIG["lr_head"]},
]
if not CONFIG["freeze_encoders"]:
    parameter_groups.append(
        {"params": model.audio.parameters(), "lr": CONFIG["lr_encoder"]}
    )
    parameter_groups.append(
        {"params": model.sheet.parameters(), "lr": CONFIG["lr_encoder"]}
    )

optimizer = torch.optim.Adam(
    parameter_groups,
    weight_decay=CONFIG["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

print("Criterion:", type(criterion).__name__)
print("Optimizer groups:", [group["lr"] for group in optimizer.param_groups])


In [ ]:
def train_one_epoch_task2(model, loader, optimizer, criterion, device, freeze_encoders):
    model.train()
    if freeze_encoders:
        model.sheet.eval()
        model.audio.eval()
    total_loss = 0.0
    total_items = 0
    for score_images, specs, labels in loader:
        score_images = score_images.to(device, non_blocking=True)
        specs = specs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(score_images, specs)
        loss = criterion(logits, labels)
        if not torch.isfinite(loss):
            raise RuntimeError("Loss Task 2 tidak finite; cek learning rate/data.")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            CONFIG["grad_clip"],
        )
        optimizer.step()
        total_loss += float(loss.detach()) * labels.size(0)
        total_items += labels.size(0)
    return total_loss / max(total_items, 1)


@torch.no_grad()
def predict_task2(model, loader, device):
    model.eval()
    probabilities = []
    labels_all = []
    for score_images, specs, labels in loader:
        logits = model(score_images.to(device), specs.to(device))
        probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())
        labels_all.append(labels.numpy())
    return np.concatenate(probabilities, axis=0), np.concatenate(labels_all, axis=0)


def fit_task2(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterion,
    config,
    device,
    checkpoint_path,
):
    history = []
    best_f1 = -1.0
    bad_epochs = 0
    for epoch in range(1, int(config["epochs"]) + 1):
        started = time.time()
        train_loss = train_one_epoch_task2(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
            config["freeze_encoders"],
        )
        probabilities, labels = predict_task2(model, val_loader, device)
        if not np.isfinite(probabilities).all():
            print("Probabilitas validasi tidak finite; training dihentikan.")
            break
        predictions = probabilities.argmax(axis=1)
        metrics = classification_metrics(
            labels,
            predictions,
            class_names=CLASS_NAMES,
            num_classes=config["num_classes"],
        )
        metrics["top2_accuracy"] = topk_accuracy(probabilities, labels, k=2)
        if not np.isfinite(metrics["macro_f1"]):
            print("Val macro-F1 tidak finite; training dihentikan.")
            break
        scheduler.step(metrics["macro_f1"])
        elapsed = time.time() - started
        record = {
            "epoch": epoch,
            "train_loss": train_loss,
            **metrics,
            "lr_head": float(optimizer.param_groups[0]["lr"]),
            "seconds": elapsed,
        }
        history.append(record)
        print(
            f"epoch {epoch:02d} | loss {train_loss:.4f} | "
            f"val acc {metrics['accuracy']:.4f} | "
            f"val macro-F1 {metrics['macro_f1']:.4f} | "
            f"top2 {metrics['top2_accuracy']:.4f} | {elapsed:.1f}s"
        )
        if metrics["macro_f1"] > best_f1:
            best_f1 = metrics["macro_f1"]
            bad_epochs = 0
            checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "config": config,
                    "config_id": CONFIG_ID,
                    "fusion": config["fusion"],
                    "freeze_encoders": config["freeze_encoders"],
                    "data_mode": config["data_mode"],
                    "task1_checkpoint": str(TASK1_CKPT),
                    "source_checkpoint": str(SOURCE_CKPT) if SOURCE_CKPT else None,
                    "epoch": epoch,
                    "val_metrics": metrics,
                },
                checkpoint_path,
            )
            print(f"  -> checkpoint disimpan (best val macro-F1 {best_f1:.4f})")
        else:
            bad_epochs += 1
            if bad_epochs >= int(config["early_stop_patience"]):
                print(f"Early stop setelah {bad_epochs} epoch tanpa perbaikan.")
                break
    return history, best_f1


In [ ]:
def plot_training_curves(history_df, path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("cross-entropy")
    axes[0].set_title("Training loss")
    axes[1].plot(history_df["epoch"], history_df["accuracy"], marker="o", label="accuracy")
    axes[1].plot(history_df["epoch"], history_df["macro_f1"], marker="o", label="macro-F1")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("score")
    axes[1].set_title("Validation metrics")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()


history = []
best_f1 = float("-inf")
history_df = pd.DataFrame()

if CONFIG_ID == "C1" and REUSE_C1:
    print("Reusing T2-baseline_seed42; tidak melakukan training.")
    checkpoint = load_checkpoint(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    best_f1 = float((checkpoint.get("val_metrics") or {}).get("macro_f1", float("nan")))
    baseline_history = ROOT / "reports" / "experiments" / "T2-baseline_seed42" / "history.csv"
    if baseline_history.exists():
        history_df = pd.read_csv(baseline_history)
        history = history_df.to_dict(orient="records")
        history_df.to_csv(HISTORY_PATH, index=False)
    save_json(
        SOURCE_PATH,
        {
            "source_run": "T2-baseline_seed42",
            "source_checkpoint": str(BASELINE_CKPT),
            "note": "T2-C1 reuses the existing baseline checkpoint.",
        },
    )
else:
    run_epochs = 2 if FAST_DEV else int(CONFIG["epochs"])
    run_patience = 2 if FAST_DEV else int(CONFIG["early_stop_patience"])
    fit_config = dict(CONFIG)
    fit_config["epochs"] = run_epochs
    fit_config["early_stop_patience"] = run_patience
    history, best_f1 = fit_task2(
        model,
        train_loader,
        val_loader,
        optimizer,
        scheduler,
        criterion,
        fit_config,
        DEVICE,
        CHECKPOINT_PATH,
    )
    history_df = pd.DataFrame(history)
    if not history_df.empty:
        history_df.drop(
            columns=["confusion_matrix", "per_class"], errors="ignore"
        ).to_csv(HISTORY_PATH, index=False)
        display(history_df.drop(columns=["confusion_matrix", "per_class"], errors="ignore"))
        plot_training_curves(history_df, FIG_DIR / "training_curves.png")

save_json(CONFIG_PATH, CONFIG)
print(f"Best validation macro-F1: {best_f1:.4f}")
print("Checkpoint:", CHECKPOINT_PATH)


In [ ]:
def evaluate_task2_loader(model, loader, device):
    probabilities, labels = predict_task2(model, loader, device)
    predictions = probabilities.argmax(axis=1)
    metrics = classification_metrics(
        labels,
        predictions,
        class_names=CLASS_NAMES,
        num_classes=CONFIG["num_classes"],
    )
    metrics["top2_accuracy"] = topk_accuracy(probabilities, labels, k=2)
    metrics["macro_precision"] = float(
        np.mean([value["precision"] for value in metrics["per_class"].values()])
    )
    metrics["macro_recall"] = float(
        np.mean([value["recall"] for value in metrics["per_class"].values()])
    )
    binary = binary_metrics(labels, predictions)

    error_mask = labels != 0
    detected_mask = predictions != 0
    both_error = error_mask & detected_mask
    error_type = classification_metrics(
        labels[both_error] - 1,
        predictions[both_error] - 1,
        class_names=CLASS_NAMES[1:],
        num_classes=3,
    )
    error_type["coverage_of_true_errors"] = float(
        both_error.sum() / max(error_mask.sum(), 1)
    )
    return {
        "metrics": metrics,
        "binary": binary,
        "error_type": error_type,
        "probabilities": probabilities,
        "labels": labels,
        "predictions": predictions,
    }


def result_summary(result):
    return {
        key: value
        for key, value in result.items()
        if key not in {"probabilities", "labels", "predictions"}
    }


def plot_confusion(confusion, title, path):
    confusion = np.asarray(confusion)
    fig, axis = plt.subplots(figsize=(6, 5))
    image = axis.imshow(confusion, cmap="Blues")
    axis.set_xticks(range(len(CLASS_NAMES)))
    axis.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
    axis.set_yticks(range(len(CLASS_NAMES)))
    axis.set_yticklabels(CLASS_NAMES)
    axis.set_xlabel("predicted")
    axis.set_ylabel("true")
    axis.set_title(title)
    threshold = confusion.max() / 2.0 if confusion.max() else 0.0
    for row in range(confusion.shape[0]):
        for column in range(confusion.shape[1]):
            axis.text(
                column,
                row,
                str(confusion[row, column]),
                ha="center",
                va="center",
                color="white" if confusion[row, column] > threshold else "black",
            )
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()


def save_predictions(meta, result, path):
    probabilities = result["probabilities"]
    labels = result["labels"]
    predictions = result["predictions"]
    if len(meta) != len(labels):
        raise ValueError("Metadata dan prediksi tidak sejajar.")
    out = meta.copy().reset_index(drop=True)
    out["prediction"] = predictions
    out["prediction_name"] = [CLASS_NAMES[int(index)] for index in predictions]
    out["prob_true"] = probabilities[np.arange(len(labels)), labels]
    out["correct"] = predictions == labels
    out.to_csv(path, index=False)
    print("Disimpan:", path)


checkpoint = load_checkpoint(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print("Loaded epoch:", checkpoint.get("epoch"))
print("Best val metrics:", checkpoint.get("val_metrics"))

full_result = evaluate_task2_loader(model, test_full_loader, DEVICE)
pitch_result = evaluate_task2_loader(model, test_pitch_loader, DEVICE)

save_predictions(test_meta_full, full_result, REPORTS_DIR / "test_predictions_full.csv")
save_predictions(test_meta_pitch, pitch_result, REPORTS_DIR / "test_predictions_pitch.csv")

plot_confusion(
    full_result["metrics"]["confusion_matrix"],
    f"{RUN_ID} - full test",
    FIG_DIR / "confusion_full.png",
)
plot_confusion(
    pitch_result["metrics"]["confusion_matrix"],
    f"{RUN_ID} - pitch-holdout test",
    FIG_DIR / "confusion_pitch.png",
)

metrics_payload = {
    "task": "Task 2 experiments",
    "config_id": CONFIG_ID,
    "config": CONFIG,
    "seed": SEED,
    "device": str(DEVICE),
    "checkpoint": str(CHECKPOINT_PATH),
    "source_run": SOURCE_RUN,
    "best_checkpoint_epoch": checkpoint.get("epoch"),
    "best_val_metrics": checkpoint.get("val_metrics"),
    "full_test": result_summary(full_result),
    "pitch_test": result_summary(pitch_result),
    "history": history,
}
save_json(METRICS_PATH, metrics_payload)
print("Disimpan:", METRICS_PATH)

print("\nFull test:")
print("  accuracy :", round(full_result["metrics"]["accuracy"], 4))
print("  macro-F1 :", round(full_result["metrics"]["macro_f1"], 4))
print("  macro-P  :", round(full_result["metrics"]["macro_precision"], 4))
print("  macro-R  :", round(full_result["metrics"]["macro_recall"], 4))
print("\nPitch-holdout test:")
print("  accuracy :", round(pitch_result["metrics"]["accuracy"], 4))
print("  macro-F1 :", round(pitch_result["metrics"]["macro_f1"], 4))
print("  macro-P  :", round(pitch_result["metrics"]["macro_precision"], 4))
print("  macro-R  :", round(pitch_result["metrics"]["macro_recall"], 4))

display(pd.DataFrame(full_result["metrics"]["per_class"]).T)
display(pd.DataFrame(pitch_result["metrics"]["per_class"]).T)


In [ ]:
def load_t2_metrics(config_id):
    path = ROOT / "reports" / "experiments" / f"T2-{config_id}_seed{SEED}" / "metrics.json"
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

rows = []
for config_id in ["C1", "C2", "C3", "C4"]:
    payload = load_t2_metrics(config_id)
    if payload is None:
        rows.append({"config": config_id, "status": "missing"})
        continue
    full = payload.get("full_test", {}).get("metrics", {})
    pitch = payload.get("pitch_test", {}).get("metrics", {})
    best_val = payload.get("best_val_metrics") or {}
    rows.append({
        "config": config_id,
        "status": "ok",
        "best_epoch": payload.get("best_checkpoint_epoch"),
        "best_val_macro_f1": best_val.get("macro_f1"),
        "full_accuracy": full.get("accuracy"),
        "full_macro_f1": full.get("macro_f1"),
        "full_macro_precision": full.get("macro_precision"),
        "full_macro_recall": full.get("macro_recall"),
        "pitch_accuracy": pitch.get("accuracy"),
        "pitch_macro_f1": pitch.get("macro_f1"),
        "pitch_macro_precision": pitch.get("macro_precision"),
        "pitch_macro_recall": pitch.get("macro_recall"),
        "pitch_n": pitch.get("n"),
    })

comparison = pd.DataFrame(rows)
display(comparison)
comparison.to_csv(
    ROOT / "reports" / "experiments" / "task2_config_comparison.csv",
    index=False,
)
print("Disimpan:", ROOT / "reports" / "experiments" / "task2_config_comparison.csv")


## Catatan

- `T2-C1` memakai ulang checkpoint `T2-baseline_seed42` bila `REUSE_C1=True`; artefak evaluasi disimpan di `reports/experiments/T2-C1_seed42/`.
- `T2-C2` membekukan kedua encoder dan hanya melatih classifier head. Encoder tetap berada dalam mode `eval()` saat training.
- `T2-C3` memakai fusion sederhana `concat([x, y])` berukuran `64-d`, lalu `128 -> 4`.
- `T2-C4` melatih pada `|pitch_delta| <= 6`, validasi `7-8`, dan menguji `9-12`; kelas clean/delete/insert tidak difilter.
- Test set penuh dan pitch-holdout test set selalu dievaluasi agar C1-C4 dapat dibandingkan pada data yang sama.
- Artefak utama: `config.json`, `history.csv`, `metrics.json`, `test_predictions_full.csv`, `test_predictions_pitch.csv`, dan `figures/`.
